# 🕉️ Sadhana Nandadeep — Data Pipeline

This notebook is the **single entry point** for processing new videos end-to-end:

1. **Environment Setup** — Mount Drive, install packages, clone repo
2. **Data Sync** *(optional)* — Pull latest edits from DynamoDB back to local files
3. **Transcript Fetch** — Download YouTube Marathi transcripts
4. **AI Enrichment** — Extract metadata with Gemini AI
5. **Metadata Cleanup** — Rule-based normalization of stories & music segments
6. **DynamoDB Upload** — Write structured metadata to `sadhananandadeep-metadata`
7. **Books Pipeline** *(optional)* — Process PDFs into metadata
8. **GPU Embedding** — Generate BGE-M3 dense vectors
9. **Qdrant Upload** — Upload hybrid (dense + sparse) vectors
10. **Verification** — Confirm output counts

> **Runtime**: Change to **T4 GPU** via `Runtime → Change runtime type`.  
> **Secrets**: Add all API keys in Colab's 🔑 Secrets sidebar before running.

---

## Part 1 — Environment Setup

Run these three cells **once** at the start of every session. They are all idempotent.

### Cell 1 — Mount Google Drive & Create Folders

Mounts Google Drive and creates the permanent storage folders.

All pipeline data (transcripts, metadata, embeddings) lives on Drive so nothing is lost when the Colab session ends.

| Property | Value |
|----------|-------|
| ⚡ **Idempotent?** | ✅ Yes — safe to re-run every session |
| 📂 **Drive path** | `MyDrive/SadhanaNandadeep_Data/` |

In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define the permanent Drive folder structure
DRIVE_ROOT = '/content/drive/MyDrive/SadhanaNandadeep_Data'
DRIVE_OUTPUT = os.path.join(DRIVE_ROOT, 'output')
DRIVE_META = os.path.join(DRIVE_ROOT, 'enriched_metadata')
DRIVE_JSON = os.path.join(DRIVE_ROOT, 'enriched_json')
DRIVE_BOOKS_INPUT = os.path.join(DRIVE_ROOT, 'input_books')
DRIVE_BOOKS_OUTPUT = os.path.join(DRIVE_ROOT, 'books_output')
DRIVE_BOOKS_META = os.path.join(DRIVE_ROOT, 'books_enriched_metadata')
DRIVE_BOOKS_CHUNKS = os.path.join(DRIVE_ROOT, 'processed_books_chunks')

# 3. Create folders if they don't exist
for d in [DRIVE_ROOT, DRIVE_OUTPUT, DRIVE_META, DRIVE_JSON, DRIVE_BOOKS_INPUT, DRIVE_BOOKS_OUTPUT, DRIVE_BOOKS_META, DRIVE_BOOKS_CHUNKS]:
    os.makedirs(d, exist_ok=True)

print(f"✅ Google Drive mounted! Data will be stored in: {DRIVE_ROOT}")


### Cell 2 — Install Dependencies & Load API Keys

Installs all required Python packages and loads API credentials from Colab Secrets.

**Required secrets** (add in the 🔑 Secrets sidebar):

| Secret | Used By |
|--------|---------|
| `GITHUB_TOKEN` | Cloning the private repo |
| `GEMINI_API_KEY` | AI enrichment (Cell 5) |
| `AWS_ACCESS_KEY_ID` | DynamoDB upload (Cell 7) |
| `AWS_SECRET_ACCESS_KEY` | DynamoDB upload (Cell 7) |
| `AWS_DEFAULT_REGION` | DynamoDB upload (Cell 7) |
| `QDRANT_URL` | Qdrant vector upload (Cell 9) |
| `QDRANT_API_KEY` | Qdrant vector upload (Cell 9) |
| `HF_API_KEY` | HuggingFace embeddings API |

| Property | Value |
|----------|-------|
| ⚡ **Idempotent?** | ✅ Yes — `pip install -q` skips already-installed packages |

In [ ]:
!apt-get update -qq && apt-get install -y -qq tesseract-ocr tesseract-ocr-mar
!pip install -q \
    "google-genai>=1.66.0,<2.0.0" \
    langchain==1.3.1 \
    langchain-text-splitters==1.1.2 \
    langchain-core==1.4.0 \
    sentence-transformers \
    torch \
    qdrant-client==1.18.0 \
    youtube-transcript-api \
    python-dotenv \
    tenacity \
    boto3 \
    PyMuPDF \
    pytesseract \
    Pillow

import os
from google.colab import userdata

# Load secrets into environment (ensure these are added in Colab's 🔑 sidebar)
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
os.environ["QDRANT_URL"]     = userdata.get("QDRANT_URL")
os.environ["QDRANT_API_KEY"] = userdata.get("QDRANT_API_KEY")
os.environ["AWS_ACCESS_KEY_ID"]     = userdata.get("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"]    = userdata.get("AWS_DEFAULT_REGION")

print("✅ Dependencies installed and secrets loaded")


### Cell 3 — Clone Repository & Create Symlinks

Clones the **latest code** from GitHub and creates symbolic links so every script reads/writes directly to Google Drive.

> ⚠️ This cell **deletes** the previous `/content/repo` clone each run. That is intentional — it ensures you always have the latest pipeline code without stale `.pyc` caches.

| Property | Value |
|----------|-------|
| ⚡ **Idempotent?** | ✅ Yes — safe to re-run |
| 🔗 **Symlinks created** | `output/`, `enriched_metadata/`, `enriched_json/`, `input_books/`, `books_output/`, `books_enriched_metadata/`, `processed_books_chunks/` |

In [ ]:
import os
from google.colab import userdata

%cd /content
!rm -rf /content/repo

# Clone Repo using the token from the Colab kernel
token = userdata.get("GITHUB_TOKEN")
!git clone https://{token}@github.com/ameyk2004/multimodal-video-search.git /content/repo --quiet

# Create symlinks so the code writes directly to Google Drive
!rm -rf /content/repo/data_pipeline/videos/output
!rm -rf /content/repo/data_pipeline/videos/enriched_metadata
!rm -rf /content/repo/data_pipeline/videos/enriched_json
!rm -rf /content/repo/data_pipeline/books/input_books
!rm -rf /content/repo/data_pipeline/books/books_output
!rm -rf /content/repo/data_pipeline/books/books_enriched_metadata
!rm -rf /content/repo/data_pipeline/books/processed_books_chunks

!ln -s /content/drive/MyDrive/SadhanaNandadeep_Data/output /content/repo/data_pipeline/videos/output
!ln -s /content/drive/MyDrive/SadhanaNandadeep_Data/enriched_metadata /content/repo/data_pipeline/videos/enriched_metadata
!ln -s /content/drive/MyDrive/SadhanaNandadeep_Data/enriched_json /content/repo/data_pipeline/videos/enriched_json
!ln -s /content/drive/MyDrive/SadhanaNandadeep_Data/input_books /content/repo/data_pipeline/books/input_books
!ln -s /content/drive/MyDrive/SadhanaNandadeep_Data/books_output /content/repo/data_pipeline/books/books_output
!ln -s /content/drive/MyDrive/SadhanaNandadeep_Data/books_enriched_metadata /content/repo/data_pipeline/books/books_enriched_metadata
!ln -s /content/drive/MyDrive/SadhanaNandadeep_Data/processed_books_chunks /content/repo/data_pipeline/books/processed_books_chunks

# Install package
!pip install -q -e /content/repo

print("✅ Repo cloned and Google Drive symlinks created.")


---
## Part 2 — Data Sync & Extraction (CPU)

These cells fetch transcripts, optionally sync edits from DynamoDB, extract metadata with AI, and upload to DynamoDB.  
No GPU required for this section.

### Cell 3.5 — (Optional) Sync from DynamoDB → Local Files

**When to run:** After you have edited stories or musical-segment timestamps in the **Admin Panel**.  
Run this cell *before* Cell 5 (AI Enrichment) so you never overwrite correct DynamoDB data with stale local files.

**What it does:**
- Queries `sadhananandadeep-metadata` via **GSI1** (`GSI1PK = "VIDEOS"`) to list every video
- For each video, queries **GSI2** (`video_id`) to fetch all `STORY#` and `MUSIC#` items
- Reconstructs stories & musical segments into the flat `_meta.json` format and writes to local Drive

| Property | Value |
|----------|-------|
| ⚡ **Idempotent?** | ✅ Yes — only updates files where DynamoDB differs from local |
| ☁️ **Direction** | DynamoDB → local `enriched_metadata/` files |
| 🔑 **Requires** | AWS credentials loaded in Cell 2 |

In [ ]:
%cd /content/repo

# Paste the contents of data_pipeline/colab/cell_3_5_sync_from_dynamo.py here,
# OR run it directly:
import os
exec(open("data_pipeline/colab/cell_3_5_sync_from_dynamo.py").read())


### Cell 4 — Fetch Raw YouTube Transcripts

Downloads fine-grained (2-5 second) Marathi transcripts for the video URLs defined in `data_pipeline/main.py`.

| Property | Value |
|----------|-------|
| ⚡ **Idempotent?** | ✅ Yes — **skips** any video whose transcript JSON already exists in `data_pipeline/videos/output/`. |
| ⚠️ **IP Block Risk** | YouTube sometimes blocks Colab's cloud IPs. If you see `IpBlocked`, try resetting the runtime or running this step locally on your Mac instead. |
| 📄 **Output** | One `.json` file per video in `data_pipeline/videos/output/` |

In [ ]:
%cd /content/repo
!python data_pipeline/videos/main.py


### Cell 5 — Fix Timestamps on Existing Metadata (Optional)

Re-resolves `start_time_seconds` for stories and musical segments using the raw transcripts, **without** re-calling the Gemini API.

Only needed if you've re-downloaded raw transcripts and want to patch timestamps in already-existing metadata.

| Property | Value |
|----------|-------|
| ⚡ **Idempotent?** | ✅ Yes — only updates items where the timestamp changed by more than 0.5 seconds. |
| 🧠 **Gemini calls?** | ❌ None — purely local computation. |
| 🔧 **Dry run** | Add `--dry-run` flag to preview changes without writing files. |

In [ ]:
%cd /content/repo
!python scripts/metadata/fix_timestamps.py


### Cell 6 — AI Video Enrichment (Gemini)

Sends each video's full transcript to Gemini to extract structured metadata: topics, queries, stories, actionable practices, quoted verses, and musical segments.

| Property | Value |
|----------|-------|
| ⚡ **Idempotent?** | ✅ Yes — **skips** any video whose `_meta.json` already exists in `data_pipeline/videos/enriched_metadata/`. |
| ⏱️ **Duration** | ~5-10 seconds per new video (Gemini API + rate limiting delay). |
| 💰 **Cost** | Uses Gemini API quota. Already-processed videos are free (skipped). |
| 📄 **Output** | One `<video_id>_meta.json` per video in `data_pipeline/videos/enriched_metadata/` |

In [ ]:
%cd /content/repo
!python -m data_pipeline.videos.video_enricher


### Cell 6.5 — Clean & Normalize Metadata

Applies hardcoded rule-based corrections to the Gemini output:
1. Cleans up hallucinated musical segments and standardizes bhajan titles.
2. Normalizes saint names to standard Marathi representations.
3. Adds English transliterations for search compatibility.

| Property | Value |
|----------|-------|
| ⚡ **Idempotent?** | ✅ Yes — updates local `_meta.json` files safely. |

In [ ]:
%cd /content/repo
!python scripts/metadata/clean_musical_segments.py
!python scripts/metadata/normalize_saint_names.py
!python scripts/metadata/add_english_transliterations.py


### Cell 7 — Upload Metadata to DynamoDB (New Schema)

Uploads enriched metadata from all local `*_meta.json` files to the **new single-table** DynamoDB schema (`sadhananandadeep-metadata`).

**What gets written per video:**

| DynamoDB Item | Key Pattern | Contents |
|---------------|-------------|---------|
| Video metadata | `VIDEO#<id> / METADATA` | title, topics, queries, practices, verses |
| Story | `SAINT#<name> / STORY#<uuid>` | title, moral, timestamps, saint |
| Music segment | `SAINT#<name> / MUSIC#<uuid>` | name, type, saint, timestamps |
| Saint profile | `SAINT#<name> / METADATA` | Created only if one doesn't already exist |

**Why your existing data is safe:**

| Item type | Behaviour on re-run |
|-----------|-------------------|
| `VIDEO# / METADATA` | Safe upsert — overwrites with latest enriched data |
| `SAINT# / METADATA` | `attribute_not_exists(PK)` — never overwrites hand-crafted bios |
| `STORY#` / `MUSIC#` items | Deterministic UUID from `(video_id + start_time)` → same ID on re-run → safe overwrite, no duplicates |

| Property | Value |
|----------|-------|
| ⚡ **Idempotent?** | ✅ Yes — safe to re-run for the same video |
| ☁️ **Target table** | `sadhananandadeep-metadata` |
| 📐 **Schema** | Single-table PK/SK with GSI1 + GSI2 |

In [ ]:
%cd /content/repo
!python data_pipeline/videos/dynamo_uploader.py


---
## Part 2.5 — Books Pipeline (Optional, CPU)

Process PDF books from `data_pipeline/books/input_books/` into enriched metadata and upload to DynamoDB.  
Skip this section if you are only processing videos this session.

### Cell 7.5 — Run Books Pipeline

Processes PDF books in three stages:
1. `book_processor.py` — extracts raw text pages from PDF
2. `book_chunk_processor.py` — splits into retrieval-friendly chunks
3. `book_enricher.py` — Gemini AI extracts topics, questions, key learnings, summary
4. `data_pipeline/books/dynamo_uploader.py` — uploads `BOOK#<id>/METADATA` to `sadhananandadeep-metadata`

**Prerequisites:** Place PDFs in `data_pipeline/books/input_books/` (symlinked to Drive).

| Property | Value |
|----------|-------|
| ⚡ **Idempotent?** | ✅ Yes — skips books already present in `books_enriched_metadata/` |
| ☁️ **Target** | DynamoDB `sadhananandadeep-metadata` + Qdrant `sadhananandadeep-books` (Cell 10) |

In [ ]:
%cd /content/repo
!python data_pipeline/books/books_main.py


---
## Part 3 — GPU Embedding & Qdrant Upload

> ⚠️ **Requires T4 GPU runtime.** Go to `Runtime → Change runtime type → T4 GPU` before running these cells.

Generates dense 1024-dim BGE-M3 embeddings and uploads hybrid (dense + sparse BM25) vectors to Qdrant Cloud.

### Cell 8 — Load Embedding Model (BGE-M3 on GPU)

Loads `BAAI/bge-m3` from HuggingFace onto the T4 GPU. This is a 1 GB model — first run takes ~2 minutes to download; subsequent runs load from the HF cache in ~30 seconds.

| Property | Value |
|----------|-------|
| ⚡ **Idempotent?** | ✅ Yes — model loads from cache if already downloaded |
| 🖥️ **Requires** | T4 GPU runtime |
| 📐 **Output dims** | 1024 (dense) |

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
embedder = SentenceTransformer("BAAI/bge-m3", device=device)
print("✅ Embedder ready")


### Cell 9 — Embed Queries & Video Transcript Chunks

This cell does **two things** in sequence:

**Step 1 — Query Embeddings** (`scripts/qdrant/embed_and_upload_queries.py`):
- Reads all `queries` from `sadhananandadeep-metadata` via **GSI1** (`GSI1PK = "VIDEOS"`)
- Embeds each query using the HuggingFace BGE-M3 API
- Uploads to Qdrant collection `sadhananandadeep-queries` (skips existing IDs)

**Step 2 — Video Chunk Embeddings** (`data_pipeline/videos/TranscriptProcessor`):
- Reads raw transcript chunks from `data_pipeline/videos/output/<video_id>.json`
- Generates dense 1024-dim BGE-M3 embeddings locally on GPU
- Saves to `data_pipeline/videos/enriched_json/<video_id>_enriched.json`

| Property | Value |
|----------|-------|
| ⚡ **Idempotent?** | ✅ Yes — Qdrant skips existing point IDs; enriched_json files are skipped if they exist |
| 🖥️ **Requires** | T4 GPU (Step 2) + HF_API_KEY (Step 1) |

In [ ]:
import os
from google.colab import userdata

# ── Step 1: Embed and upload search queries ──
os.environ["HF_API_KEY"] = userdata.get("HF_API_KEY")
!cd /content/repo && python scripts/qdrant/embed_and_upload_queries.py --yes

# ── Step 2: Embed video transcript chunks on GPU ──
import sys
sys.path.append('/content/repo')

import json, glob
from data_pipeline.videos.transcript_processor import TranscriptProcessor

processor = TranscriptProcessor()
raw_files = sorted(glob.glob("/content/repo/data_pipeline/videos/output/*.json"))
total_files = len(raw_files)
print(f"\n🔢 Found {total_files} raw transcript files to process.")

skipped = 0
processed = 0

for idx, filepath in enumerate(raw_files, start=1):
    video_id = os.path.splitext(os.path.basename(filepath))[0]
    out_path = f"/content/repo/data_pipeline/videos/enriched_json/{video_id}_enriched.json"

    if os.path.exists(out_path):
        skipped += 1
        continue

    print(f"[{idx}/{total_files}] 🔄 Embedding chunks for {video_id}...")
    chunks, _, _ = processor.process_file(filepath, video_id)

    for chunk in chunks:
        chunk["embedding_vector"] = embedder.encode(chunk["marathi_raw"]).tolist()

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)
    processed += 1
    print(f"    ✅ Saved {len(chunks)} chunks")

print(f"\n{"="*50}")
print(f"📊 EMBEDDING SUMMARY")
print(f"{"="*50}")
print(f"Total raw files:     {total_files}")
print(f"Newly embedded:      {processed}")
print(f"Skipped (existing):  {skipped}")
print(f"{"="*50}")
print("✅ Query and Chunk Embedding complete!")

# ── Step 3: Embed book chunks on GPU ──
book_chunk_files = sorted(glob.glob("/content/repo/data_pipeline/books/processed_books_chunks/*_chunks.json"))
print(f"\n🔢 Found {len(book_chunk_files)} book chunk files to process.")
for filepath in book_chunk_files:
    with open(filepath, "r", encoding="utf-8") as f:
        chunks = json.load(f)
    
    needs_update = False
    print(f"🔄 Embedding chunks for {os.path.basename(filepath)}...")
    for chunk in chunks:
        if not chunk.get("embedding_vector"):
            chunk["embedding_vector"] = embedder.encode(chunk["marathi_raw"]).tolist()
            needs_update = True
    
    if needs_update:
        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(chunks, f, ensure_ascii=False, indent=2)
        print(f"    ✅ Updated {len(chunks)} chunks with embeddings")
    else:
        print(f"    ⏭️ Skipped (already embedded)")


### Cell 10 — Upload to Qdrant (Hybrid Search)

Builds BM25 sparse vocabulary from all transcript chunks, computes sparse vectors, and uploads **dense + sparse** pairs to Qdrant.

> ⚠️ **This DELETES and RECREATES the Qdrant collections** (`sadhananandadeep-videos`, `sadhananandadeep-books`) on every run. This is intentional — it ensures vectors are always consistent with the latest embeddings.

| Property | Value |
|----------|-------|
| ⚡ **Idempotent?** | ✅ (collection is rebuilt fresh each time) |
| ☁️ **Target** | Qdrant Cloud collections `sadhananandadeep-videos` + `sadhananandadeep-books` |
| 🖥️ **Requires** | `enriched_json/` files from Cell 9 |

In [ ]:
%cd /content/repo
!python scripts/qdrant/rebuild_hybrid_collection.py
!python scripts/qdrant/rebuild_books_collection.py


---
## Part 4 — Verification

Quick sanity checks to confirm the pipeline ran correctly.

### Cell 11 — Verify Pipeline Output

Counts the files in each output directory to confirm processing completed for all videos.

Expected: the count in each directory should match the number of videos you intended to process.

In [ ]:
import os

dirs = {
    "Raw Transcripts (output/)": "/content/repo/data_pipeline/videos/output",
    "Enriched Metadata (enriched_metadata/)": "/content/repo/data_pipeline/videos/enriched_metadata",
    "Embedded Chunks (enriched_json/)": "/content/repo/data_pipeline/videos/enriched_json",
    "Books Input (input_books/)": "/content/repo/data_pipeline/books/input_books",
    "Books Output (books_output/)": "/content/repo/data_pipeline/books/books_output",
    "Books Enriched Metadata (books_enriched_metadata/)": "/content/repo/data_pipeline/books/books_enriched_metadata",
    "Books Chunks (processed_books_chunks/)": "/content/repo/data_pipeline/books/processed_books_chunks",
}

print("=" * 50)
print("📊 PIPELINE OUTPUT VERIFICATION")
print("=" * 50)

for label, path in dirs.items():
    if os.path.exists(path):
        files = [f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))]
        print(f"{label}: {len(files)} files")
    else:
        print(f"{label}: ❌ Directory not found")

print("=" * 50)
print("\n🎉 Pipeline verification complete!")


---
## Quick Reference — Cell Execution Order

| Step | Cell | Description | Skip if... |
|------|------|-------------|-----------|
| 1 | **Cell 1** | Mount Drive | — |
| 2 | **Cell 2** | Install packages & load secrets | — |
| 3 | **Cell 3** | Clone repo & create symlinks | — |
| 3.5 | **Cell 3.5** | *(Optional)* Sync DynamoDB → local files | Not editing via Admin Panel |
| 4 | **Cell 4** | Fetch YouTube transcripts | Already downloaded |
| 5 | **Cell 5** | *(Optional)* Fix timestamps | Not re-downloading transcripts |
| 6 | **Cell 6** | AI enrichment with Gemini | Already enriched |
| 6.5 | **Cell 6.5** | Clean & normalize metadata | — |
| 7 | **Cell 7** | Upload metadata to DynamoDB | — |
| 7.5 | **Cell 7.5** | *(Optional)* Books pipeline | Not processing books |
| 8 | **Cell 8** | Load BGE-M3 model on GPU | — |
| 9 | **Cell 9** | Embed queries & video chunks | — |
| 10 | **Cell 10** | Upload vectors to Qdrant | — |
| 11 | **Cell 11** | Verify output | — |

---
*Pipeline notebook — migrated to new DynamoDB single-table schema, June 2026.*